In [ ]:
import pandas as pd
import pint
import pint_xarray
import xarray as xr

from imagematerials.constants import _IMAGE_REGIONS as image_regions
from util import create_region_graph
from constants import target_regions

In [10]:
ureg = pint.UnitRegistry()
ureg.define('terakilometer = 1e12 * kilometer')
ureg.define('milkilometer = 1e6 * kilometer')

region_mapping = { i+1:region for i, region in enumerate(image_regions)}

pkmGlobal_original = pd.read_csv("image_3_4_data/pkmGlobalTot.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)
tkmGlobal_original = pd.read_csv("image_3_4_data/tkmGlobalTot.csv", delimiter=";", usecols=range(8),skiprows=3).drop(0)
load_original = pd.read_csv("image_3_4_data/load.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)

In [11]:
def convert_to_xarray(df, value_name, variable):
    df = df.rename(columns={"Unnamed: 0":"year","Unnamed: 1":"region"})
    df = (
        df.astype({"year": int, "region": int})  # Convert to integers
        .query("region not in [27, 28] and year in [2019, 2060]")  # Filter regions and years
    )
    # Convert all other columns (modes) to float
    df.iloc[:, 2:] = df.iloc[:, 2:].astype(float)
    df['region'] = df['region'].map(region_mapping)

    # Melt the DataFrame so that transport modes become a single coordinate
    df = df.melt(id_vars=["year", "region"], var_name= variable, value_name=value_name)

    # Convert to xarray DataArray
    dataarray = df.set_index(["region", "year", "mode"]).to_xarray()[value_name]
    return dataarray

In [12]:
pkmGlobal = convert_to_xarray(pkmGlobal_original, value_name = "pkm", variable = "mode")
tkmGlobal = convert_to_xarray(tkmGlobal_original, value_name = "tkm", variable = "mode")
load = convert_to_xarray(load_original, value_name = "load", variable = "mode")


In [13]:
# Convert units
pkmGlobal = pkmGlobal.pint.quantify('terakilometer') 
pkmGlobal= pkmGlobal.pint.to("km")

In [23]:
# Convert the region names in xr_image_output to match the target regions
knowledge_graph = create_region_graph()
xr_regions = pkmGlobal.coords["region"].values  # Extract current region names

xr_region_filter = knowledge_graph.find_relations_inverse(xr_regions, target_regions)
xr_region_filter


{'CAN': 'Canada',
 'CEU': 'Central Europe',
 'CHN': 'China',
 'INDIA': 'India',
 'JAP': 'Japan',
 'USA': 'USA',
 'WEU': 'Western Europe',
 'MEX': 'Latin America',
 'RCAM': 'Latin America',
 'BRA': 'Latin America',
 'RSAM': 'Latin America',
 'NAF': 'Middle East and Northern Africa',
 'ME': 'Middle East and Northern Africa',
 'KOR': 'Other Asia',
 'SEAS': 'Other Asia',
 'INDO': 'Other Asia',
 'RSAS': 'Other Asia',
 'OCE': 'Other OECD',
 'TUR': 'Reforming Economies',
 'UKR': 'Reforming Economies',
 'STAN': 'Reforming Economies',
 'RUS': 'Reforming Economies',
 'WAF': 'Subsaharan Africa',
 'EAF': 'Subsaharan Africa',
 'SAF': 'Subsaharan Africa',
 'RSAF': 'Subsaharan Africa'}

All image regions are included in the knowledge graph!
